# Automatic 5-Electrode Replacement Montage Suggestions

Give the notebook an original montage, such as `['F3', 'F4', 'Cz', 'CP1', 'P2']`. It keeps any original sites that are already `open`, replaces only blocked/unavailable sites, and returns ranked replacement montages.

Scope: 2D geometry only. No electric-field modeling, no 3D coordinates, no GUI features, and no file saving.

## 1. Imports

In [ ]:
try:
    import shapely
except ImportError:
    !pip install shapely

import itertools
import math
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import MultiPoint, Point
from shapely.ops import unary_union
from matplotlib.patches import Circle, Wedge, Ellipse, Polygon
from matplotlib.lines import Line2D

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 180)

## 2. Load Coordinates From Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Change this to your actual coordinate CSV path.
COORDINATE_FILE = '/content/drive/MyDrive/EEG_Project/Cap_Data/easycap_cac64_soterix_draft.csv'

def find_candidate_coordinate_files(search_root='/content/drive/MyDrive'):
    root = Path(search_root)
    if not root.exists():
        return []
    csv_files = list(root.rglob('*.csv'))
    likely_terms = ['easycap', 'soterix', 'coordinate', 'coords', 'cap']
    likely = [p for p in csv_files if any(term in p.name.lower() for term in likely_terms)]
    return likely or csv_files

def load_coords(csv_path):
    csv_path = Path(csv_path)
    if not csv_path.exists():
        candidates = find_candidate_coordinate_files()
        message = f'Coordinate CSV not found: {csv_path}'
        if candidates:
            message += '\n\nPossible CSV files found in Drive:'
            for i, candidate in enumerate(candidates[:20], start=1):
                message += f'\n{i}. {candidate}'
            message += '\n\nCopy the correct path into COORDINATE_FILE and rerun this cell.'
        raise FileNotFoundError(message)

    coords = pd.read_csv(csv_path)
    required = {'label', 'x', 'y', 'status'}
    missing = required - set(coords.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    coords = coords.copy()
    coords['label'] = coords['label'].astype(str).str.strip()
    coords['status'] = coords['status'].astype(str).str.lower().str.strip()
    return coords

coords = load_coords(COORDINATE_FILE)
print(coords['status'].value_counts())
coords.head()

## 3. Geometry Helpers

In [ ]:
DEFAULT_REPLACEMENT_WEIGHTS = {
    # The strict 85% rule now uses montage-region coverage, not circular pad overlap.
    # Footprint coverage is still included, but with much lower importance.
    'distance': 0.20,
    'pairwise_distance': 0.16,
    'pairwise_angle': 0.10,
    'region_coverage_loss': 0.36,
    'footprint_coverage_loss': 0.06,
    'new_area': 0.07,
    'center_shift': 0.05,
}

def get_xy(coords, label):
    matches = coords.loc[coords['label'] == label]
    if matches.empty:
        raise ValueError(f'Label not found in coordinate table: {label}')
    if len(matches) > 1:
        raise ValueError(f'Duplicate label in coordinate table: {label}')
    row = matches.iloc[0]
    return (float(row['x']), float(row['y']))

def status_for_label(coords, label):
    matches = coords.loc[coords['label'] == label]
    if matches.empty:
        raise ValueError(f'Label not found in coordinate table: {label}')
    return str(matches.iloc[0]['status']).lower().strip()

def labels_to_points(coords, labels):
    return [get_xy(coords, label) for label in labels]

def point_distance(p1, p2):
    return math.hypot(p1[0] - p2[0], p1[1] - p2[1])

def angle_of_vector(p1, p2):
    return math.degrees(math.atan2(p2[1] - p1[1], p2[0] - p1[0]))

def angle_difference_deg(angle1, angle2):
    diff = abs(angle1 - angle2) % 360
    return min(diff, 360 - diff)

def paired_distances_from_original(original_points, candidate_points):
    if len(original_points) != len(candidate_points):
        raise ValueError('Original and candidate point lists must have the same length')
    return [point_distance(o, c) for o, c in zip(original_points, candidate_points)]

def mean_distance_from_original(original_points, candidate_points):
    distances = paired_distances_from_original(original_points, candidate_points)
    return sum(distances) / len(distances)

def max_distance_from_original(original_points, candidate_points):
    return max(paired_distances_from_original(original_points, candidate_points))

def mean_pairwise_distance_change(original_points, candidate_points):
    if len(original_points) != len(candidate_points):
        raise ValueError('Original and candidate point lists must have the same length')
    changes = []
    for i in range(len(original_points)):
        for j in range(i + 1, len(original_points)):
            original_distance = point_distance(original_points[i], original_points[j])
            candidate_distance = point_distance(candidate_points[i], candidate_points[j])
            changes.append(abs(original_distance - candidate_distance))
    return sum(changes) / len(changes) if changes else 0

def mean_pairwise_angle_change_deg(original_points, candidate_points):
    if len(original_points) != len(candidate_points):
        raise ValueError('Original and candidate point lists must have the same length')
    changes = []
    for i in range(len(original_points)):
        for j in range(i + 1, len(original_points)):
            original_angle = angle_of_vector(original_points[i], original_points[j])
            candidate_angle = angle_of_vector(candidate_points[i], candidate_points[j])
            changes.append(angle_difference_deg(original_angle, candidate_angle))
    return sum(changes) / len(changes) if changes else 0

def montage_center(points):
    return (
        sum(point[0] for point in points) / len(points),
        sum(point[1] for point in points) / len(points),
    )

def center_shift(original_points, candidate_points):
    return point_distance(montage_center(original_points), montage_center(candidate_points))

def electrode_area(points, electrode_radius):
    circles = [Point(x, y).buffer(electrode_radius) for x, y in points]
    return unary_union(circles)

def area_overlap_metrics(original_points, candidate_points, electrode_radius=0.45):
    original_shape = electrode_area(original_points, electrode_radius)
    candidate_shape = electrode_area(candidate_points, electrode_radius)
    overlap_area = original_shape.intersection(candidate_shape).area
    original_area = original_shape.area
    candidate_area = candidate_shape.area
    new_area = candidate_area - overlap_area
    return {
        'percent_original_area_covered': (overlap_area / original_area) * 100,
        'percent_candidate_area_new': (new_area / candidate_area) * 100,
        'new_area': new_area,
    }

def montage_region(points, region_buffer=0.0):
    if len(points) < 3:
        raise ValueError('Montage region coverage requires at least 3 points')
    region = MultiPoint(points).convex_hull
    if region_buffer:
        region = region.buffer(region_buffer)
    if region.area == 0:
        region = region.buffer(1e-6)
    return region

def region_overlap_metrics(original_points, candidate_points, region_buffer=0.0):
    original_region = montage_region(original_points, region_buffer=region_buffer)
    candidate_region = montage_region(candidate_points, region_buffer=region_buffer)
    overlap_area = original_region.intersection(candidate_region).area
    original_area = original_region.area
    candidate_area = candidate_region.area
    new_region_area = candidate_area - overlap_area
    return {
        'percent_original_region_covered': (overlap_area / original_area) * 100,
        'percent_candidate_region_new': (new_region_area / candidate_area) * 100,
        'new_region_area': new_region_area,
    }

def replacement_normalized_score(
    mean_distance,
    pairwise_distance_change,
    pairwise_angle_change_deg,
    percent_original_region_covered,
    percent_original_footprint_covered,
    percent_candidate_area_new,
    center_shift_value,
    weights=None,
):
    weights = DEFAULT_REPLACEMENT_WEIGHTS if weights is None else weights
    distance_penalty = min(mean_distance / 2.0, 1)
    pairwise_distance_penalty = min(pairwise_distance_change / 2.0, 1)
    pairwise_angle_penalty = min(pairwise_angle_change_deg / 90.0, 1)
    region_coverage_loss_penalty = (100 - percent_original_region_covered) / 100
    footprint_coverage_loss_penalty = (100 - percent_original_footprint_covered) / 100
    new_area_penalty = percent_candidate_area_new / 100
    center_shift_penalty = min(center_shift_value / 2.0, 1)
    return (
        weights['distance'] * distance_penalty
        + weights['pairwise_distance'] * pairwise_distance_penalty
        + weights['pairwise_angle'] * pairwise_angle_penalty
        + weights['region_coverage_loss'] * region_coverage_loss_penalty
        + weights['footprint_coverage_loss'] * footprint_coverage_loss_penalty
        + weights['new_area'] * new_area_penalty
        + weights['center_shift'] * center_shift_penalty
    )


## 4. Automatic Replacement Functions

In [ ]:
def nearest_open_labels(coords, target_label, pool_size=10, exclude_labels=None, max_replacement_distance=None):
    exclude_labels = set(exclude_labels or [])
    target_xy = get_xy(coords, target_label)
    open_df = coords.loc[coords['status'] == 'open'].copy()
    open_df = open_df.loc[~open_df['label'].isin(exclude_labels)]
    open_df['distance_to_original'] = open_df.apply(
        lambda row: point_distance(target_xy, (float(row['x']), float(row['y']))),
        axis=1,
    )
    if max_replacement_distance is not None:
        open_df = open_df.loc[open_df['distance_to_original'] <= max_replacement_distance]
    return open_df.sort_values('distance_to_original').head(pool_size)['label'].tolist()

def score_replacement_montage(coords, original_labels, candidate_labels, electrode_radius=0.45, region_buffer=0.0, weights=None):
    if len(original_labels) != len(candidate_labels):
        raise ValueError('Original and candidate montages must have the same length')
    original_points = labels_to_points(coords, original_labels)
    candidate_points = labels_to_points(coords, candidate_labels)
    mean_distance = mean_distance_from_original(original_points, candidate_points)
    max_distance = max_distance_from_original(original_points, candidate_points)
    pairwise_distance_change = mean_pairwise_distance_change(original_points, candidate_points)
    pairwise_angle_change = mean_pairwise_angle_change_deg(original_points, candidate_points)
    area_metrics = area_overlap_metrics(original_points, candidate_points, electrode_radius=electrode_radius)
    region_metrics = region_overlap_metrics(original_points, candidate_points, region_buffer=region_buffer)
    center_shift_value = center_shift(original_points, candidate_points)
    final_score = replacement_normalized_score(
        mean_distance=mean_distance,
        pairwise_distance_change=pairwise_distance_change,
        pairwise_angle_change_deg=pairwise_angle_change,
        percent_original_region_covered=region_metrics['percent_original_region_covered'],
        percent_original_footprint_covered=area_metrics['percent_original_area_covered'],
        percent_candidate_area_new=area_metrics['percent_candidate_area_new'],
        center_shift_value=center_shift_value,
        weights=weights,
    )
    return {
        'candidate_montage': ', '.join(candidate_labels),
        'mean_distance_from_original': mean_distance,
        'max_distance_from_original': max_distance,
        'pairwise_distance_change': pairwise_distance_change,
        'pairwise_angle_change_deg': pairwise_angle_change,
        'percent_original_region_covered': region_metrics['percent_original_region_covered'],
        'percent_candidate_region_new': region_metrics['percent_candidate_region_new'],
        'new_region_area': region_metrics['new_region_area'],
        'percent_original_area_covered': area_metrics['percent_original_area_covered'],
        'percent_candidate_area_new': area_metrics['percent_candidate_area_new'],
        'new_area': area_metrics['new_area'],
        'center_shift': center_shift_value,
        'final_score': final_score,
    }

def suggest_replacement_montages(coords, original_labels, top_n=10, nearby_pool_size=10, electrode_radius=0.45, region_buffer=0.0, max_replacement_distance=None, min_original_region_covered=None, weights=None):
    if not original_labels:
        raise ValueError('original_labels cannot be empty')

    for label in original_labels:
        get_xy(coords, label)

    fixed_candidate = []
    kept_originals = []
    blocked_indices = []

    for index, label in enumerate(original_labels):
        if status_for_label(coords, label) == 'open':
            fixed_candidate.append(label)
            kept_originals.append(label)
        else:
            fixed_candidate.append(None)
            blocked_indices.append(index)

    if not blocked_indices:
        row = score_replacement_montage(
            coords,
            original_labels,
            original_labels,
            electrode_radius=electrode_radius,
            region_buffer=region_buffer,
            weights=weights,
        )
        row.update({'replacements': 'none', 'kept_originals': ', '.join(kept_originals), 'blocked_originals': 'none'})
        results = pd.DataFrame([row])
        results.insert(0, 'rank', [1])
        return results

    kept_label_set = set(kept_originals)
    replacement_pools = []
    for blocked_index in blocked_indices:
        original_label = original_labels[blocked_index]
        pool = nearest_open_labels(
            coords,
            target_label=original_label,
            pool_size=nearby_pool_size,
            exclude_labels=kept_label_set,
            max_replacement_distance=max_replacement_distance,
        )
        if not pool:
            raise ValueError(f'No open replacement candidates found for {original_label}')
        replacement_pools.append(pool)

    rows = []
    for replacements in itertools.product(*replacement_pools):
        if len(set(replacements)) != len(replacements):
            continue
        if kept_label_set.intersection(replacements):
            continue

        candidate = fixed_candidate.copy()
        for blocked_index, replacement in zip(blocked_indices, replacements):
            candidate[blocked_index] = replacement
        candidate_labels = [label for label in candidate if label is not None]
        if len(set(candidate_labels)) != len(candidate_labels):
            continue

        row = score_replacement_montage(
            coords,
            original_labels,
            candidate_labels,
            electrode_radius=electrode_radius,
            region_buffer=region_buffer,
            weights=weights,
        )
        if min_original_region_covered is not None and row['percent_original_region_covered'] < min_original_region_covered:
            continue
        row.update({
            'replacements': '; '.join(f'{original_labels[index]}->{replacement}' for index, replacement in zip(blocked_indices, replacements)),
            'kept_originals': ', '.join(kept_originals) if kept_originals else 'none',
            'blocked_originals': ', '.join(original_labels[index] for index in blocked_indices),
        })
        rows.append(row)

    if not rows:
        if min_original_region_covered is not None:
            raise ValueError(f'No valid replacement montages were generated that cover at least {min_original_region_covered}% of the original montage region')
        raise ValueError('No valid replacement montages were generated')

    results = pd.DataFrame(rows).sort_values('final_score').reset_index(drop=True)
    results.insert(0, 'rank', range(1, len(results) + 1))
    return results.head(top_n)


## 5. Enter Your 5 Original Placements

Blocked sites will be replaced. Open sites will be kept.

In [ ]:
original_labels = ['CP4', 'P2', 'P6', 'PO8', 'O2']

status_table = pd.DataFrame([
    {'label': label, 'status': status_for_label(coords, label)}
    for label in original_labels
])
status_table

## 6. Generate Suggestions

## 6a. Tune Scoring Weights

Increase `region_coverage_loss` if the replacement montage should enclose the same overall region. `footprint_coverage_loss` is the older circular pad-overlap metric and is intentionally lower importance.

In [ ]:
replacement_weights = {
    'distance': 0.20,
    'pairwise_distance': 0.16,
    'pairwise_angle': 0.10,
    'region_coverage_loss': 0.36,
    'footprint_coverage_loss': 0.06,
    'new_area': 0.07,
    'center_shift': 0.05,
}

print('Weight total:', round(sum(replacement_weights.values()), 3))
replacement_weights


In [ ]:
suggestions_df = suggest_replacement_montages(
    coords=coords,
    original_labels=original_labels,
    top_n=10,
    nearby_pool_size=10,
    electrode_radius=0.45,
    region_buffer=0.0,
    max_replacement_distance=None,
    min_original_region_covered=None,
    weights=replacement_weights,
)

display_columns = [
    'rank',
    'candidate_montage',
    'replacements',
    'kept_originals',
    'percent_original_region_covered',
    'percent_candidate_region_new',
    'percent_original_area_covered',
    'percent_candidate_area_new',
    'mean_distance_from_original',
    'pairwise_distance_change',
    'pairwise_angle_change_deg',
    'center_shift',
    'final_score',
]

suggestions_df[display_columns]


## 6b. Strict 85% Region-Coverage Suggestions

This version only considers replacement montages valid if the candidate convex-hull montage region covers at least 85% of the original convex-hull montage region. This is different from circular electrode footprint overlap.

In [ ]:
min_original_region_covered = 85

try:
    strict_suggestions_df = suggest_replacement_montages(
        coords=coords,
        original_labels=original_labels,
        top_n=10,
        nearby_pool_size=10,
        electrode_radius=0.45,
        region_buffer=0.0,
        max_replacement_distance=None,
        min_original_region_covered=min_original_region_covered,
        weights=replacement_weights,
    )
    strict_suggestions_df[display_columns]
except ValueError as exc:
    print(exc)
    strict_suggestions_df = pd.DataFrame()


## 7. Inspect Best Option

Lower `final_score` is better. The table remains in memory as `suggestions_df`; nothing is saved to Drive.

In [ ]:
best = suggestions_df.iloc[0]
print('Best candidate montage:')
print(best['candidate_montage'])
print('\nReplacements:')
print(best['replacements'])
print('\nKept originals:')
print(best['kept_originals'])
print('\nFull score row:')
display(best.to_frame().T)

## 8. Visualize Original vs Suggested Montage

Color key: original placements are green, suggested placements are purple, sites in both are half green/half purple, unavailable EEG slots are red, and unused open Soterix slots are white.

In [ ]:
def parse_montage_labels(montage_text):
    return [label.strip() for label in str(montage_text).split(',') if label.strip()]

def apply_layout_tweaks_for_plot(df):
    plot_df = df.copy()

    lateral_blocked = ['FT7', 'FT8', 'T7', 'T8', 'TP7', 'TP8', 'P7', 'P8', 'F7', 'F8']
    plot_df.loc[plot_df['label'].isin(lateral_blocked), 'x'] *= 0.96

    lower_posterior = ['PO7', 'PO8', 'PO9', 'PO10', 'P9', 'P10', 'PPO9h', 'PPO10h', 'POO9h', 'POO10h']
    plot_df.loc[plot_df['label'].isin(lower_posterior), 'y'] += 0.10

    posterior_open = ['P9', 'P10', 'PPO9h', 'PPO10h', 'POO9h', 'POO10h']
    plot_df.loc[plot_df['label'].isin(posterior_open), 'x'] *= 0.97

    plot_df.loc[plot_df['label'].isin(['O1', 'O2']), 'y'] += 0.12
    return plot_df

def plot_original_vs_suggestion(coords, original_labels, suggestion_row, title='Original vs suggested montage'):
    suggested_labels = parse_montage_labels(suggestion_row['candidate_montage'])
    original_set = set(original_labels)
    suggested_set = set(suggested_labels)
    both_set = original_set & suggested_set

    plot_df = apply_layout_tweaks_for_plot(coords)
    fig, ax = plt.subplots(figsize=(9, 9))

    head = Ellipse((0, 0.25), width=10.25, height=9.35, facecolor='none', edgecolor='#4b5563', linewidth=1.2, zorder=0)
    nose = Polygon([[-0.36, 4.95], [0, 5.65], [0.36, 4.95]], closed=True, facecolor='none', edgecolor='#4b5563', linewidth=1.2, zorder=0)
    ax.add_patch(head)
    ax.add_patch(nose)

    for row in plot_df.itertuples():
        label = row.label
        is_original = label in original_set
        is_suggested = label in suggested_set
        is_both = label in both_set

        if is_both:
            ax.add_patch(Wedge((row.x, row.y), 0.33, 90, 270, facecolor='#22c55e', edgecolor='#111827', linewidth=1.1, zorder=4))
            ax.add_patch(Wedge((row.x, row.y), 0.33, -90, 90, facecolor='#a855f7', edgecolor='#111827', linewidth=1.1, zorder=4))
        elif is_original:
            ax.add_patch(Circle((row.x, row.y), 0.33, facecolor='#22c55e', edgecolor='#166534', linewidth=1.4, zorder=4))
        elif is_suggested:
            ax.add_patch(Circle((row.x, row.y), 0.33, facecolor='#a855f7', edgecolor='#6b21a8', linewidth=1.4, zorder=4))
        elif row.status == 'blocked':
            ax.add_patch(Circle((row.x, row.y), 0.28, facecolor='#ef4444', edgecolor='#991b1b', linewidth=1.0, zorder=2))
        else:
            ax.add_patch(Circle((row.x, row.y), 0.28, facecolor='white', edgecolor='#667085', linewidth=1.0, linestyle='--', zorder=1))

        label_color = 'white' if (row.status == 'blocked' and not is_original and not is_suggested) else '#111827'
        label_weight = 'bold' if (is_original or is_suggested) else 'normal'
        ax.text(row.x, row.y, label, ha='center', va='center', fontsize=7.2, color=label_color, fontweight=label_weight, zorder=5)

    # Draw replacement arrows from blocked original sites to suggested replacement sites.
    points = plot_df.set_index('label')
    for original_label, suggested_label in zip(original_labels, suggested_labels):
        if original_label == suggested_label:
            continue
        if original_label not in points.index or suggested_label not in points.index:
            continue
        original_point = points.loc[original_label]
        suggested_point = points.loc[suggested_label]
        ax.annotate(
            '',
            xy=(suggested_point.x, suggested_point.y),
            xytext=(original_point.x, original_point.y),
            arrowprops=dict(arrowstyle='->', color='#6b21a8', linewidth=1.4, alpha=0.75),
            zorder=3,
        )

    legend_handles = [
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#22c55e', markeredgecolor='#166534', markersize=10, label='Original placement'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#a855f7', markeredgecolor='#6b21a8', markersize=10, label='Suggested placement'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='#ef4444', markeredgecolor='#991b1b', markersize=10, label='Unavailable EEG'),
        Line2D([0], [0], marker='o', color='none', markerfacecolor='white', markeredgecolor='#667085', markersize=10, label='Unused open slot'),
    ]
    ax.legend(handles=legend_handles, loc='upper right')
    ax.set_title(title)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlim(plot_df['x'].min() - 0.8, plot_df['x'].max() + 0.8)
    ax.set_ylim(plot_df['y'].min() - 0.8, plot_df['y'].max() + 0.9)
    ax.axis('off')
    plt.show()
    return fig, ax

plot_original_vs_suggestion(
    coords,
    original_labels,
    suggestions_df.iloc[0],
    title='Original montage vs best suggested replacement',
);


## 9. Visualize Another Ranked Option

Change `option_rank` to inspect a different suggestion from `suggestions_df`.

In [ ]:
option_rank = 2
if option_rank <= len(suggestions_df):
    plot_original_vs_suggestion(
        coords,
        original_labels,
        suggestions_df.iloc[option_rank - 1],
        title=f'Original montage vs suggestion rank {option_rank}',
    );
else:
    print(f'Only {len(suggestions_df)} suggestions are available.')


In [ ]:
manual_candidate_labels = ['POO2', 'POO10h', 'TPP8h', 'CCP6h', 'PPO2h']

manual_status_table = pd.DataFrame([
    {'label': label, 'status': status_for_label(coords, label)}
    for label in manual_candidate_labels
])

manual_status_table

In [ ]:
def score_manual_candidate(coords, original_labels, manual_candidate_labels, electrode_radius=0.45, region_buffer=0.0, weights=None):
    if len(original_labels) != len(manual_candidate_labels):
        raise ValueError('manual_candidate_labels must have the same length as original_labels')
    if len(set(manual_candidate_labels)) != len(manual_candidate_labels):
        raise ValueError('manual_candidate_labels contains duplicate positions')

    blocked_manual = [
        label for label in manual_candidate_labels
        if status_for_label(coords, label) != 'open'
    ]
    if blocked_manual:
        print('Warning: these manual candidate positions are not marked open:', blocked_manual)

    row = score_replacement_montage(
        coords=coords,
        original_labels=original_labels,
        candidate_labels=manual_candidate_labels,
        electrode_radius=electrode_radius,
        region_buffer=region_buffer,
        weights=weights,
    )

    replacements = []
    kept = []
    for original_label, candidate_label in zip(original_labels, manual_candidate_labels):
        if original_label == candidate_label:
            kept.append(original_label)
        else:
            replacements.append(f'{original_label}->{candidate_label}')

    row.update({
        'replacements': '; '.join(replacements) if replacements else 'none',
        'kept_originals': ', '.join(kept) if kept else 'none',
        'blocked_originals': ', '.join([label for label in original_labels if status_for_label(coords, label) != 'open']) or 'none',
    })
    return pd.DataFrame([row])

manual_score_df = score_manual_candidate(
    coords=coords,
    original_labels=original_labels,
    manual_candidate_labels=manual_candidate_labels,
    electrode_radius=0.45,
    region_buffer=0.0,
    weights=replacement_weights,
)

manual_display_columns = [
    'candidate_montage',
    'replacements',
    'kept_originals',
    'percent_original_region_covered',
    'percent_candidate_region_new',
    'percent_original_area_covered',
    'percent_candidate_area_new',
    'mean_distance_from_original',
    'pairwise_distance_change',
    'pairwise_angle_change_deg',
    'center_shift',
    'final_score',
]

manual_score_df[manual_display_columns]


In [ ]:
comparison_df = pd.concat([
    suggestions_df.assign(source='automatic'),
    manual_score_df.assign(source='manual'),
], ignore_index=True)

comparison_df = comparison_df.sort_values('final_score').reset_index(drop=True)
comparison_df.insert(0, 'comparison_rank', range(1, len(comparison_df) + 1))

comparison_display_columns = [
    'comparison_rank',
    'source',
    'candidate_montage',
    'replacements',
    'kept_originals',
    'percent_original_region_covered',
    'percent_candidate_region_new',
    'percent_original_area_covered',
    'percent_candidate_area_new',
    'mean_distance_from_original',
    'pairwise_distance_change',
    'pairwise_angle_change_deg',
    'center_shift',
    'final_score',
]

comparison_df[comparison_display_columns]


In [ ]:
plot_original_vs_suggestion(
    coords,
    original_labels,
    manual_score_df.iloc[0],
    title='Original montage vs manual candidate',
);


## 10. Wrapper Workflow Control Panel

This is the all-in-one version. Set the original placements, choose runtime/search settings, and rate what matters from 0 to 10. Higher importance means that metric has more influence on the final score.

In [ ]:
DEFAULT_IMPORTANCE = {
    'distance': 6,          # keep each replacement close to its original site
    'pairwise_distance': 5, # preserve spacing among the 5 electrodes
    'pairwise_angle': 4,    # preserve montage shape/orientation
    'region_coverage_loss': 10,    # preserve/enclose the same montage region
    'footprint_coverage_loss': 2,   # old circular pad-overlap metric
    'new_area': 2,                 # avoid too much new circular footprint area
    'center_shift': 2,      # avoid shifting the whole montage
}

def importance_to_weights(importance=None):
    merged = DEFAULT_IMPORTANCE.copy()
    if importance is not None:
        unknown = set(importance) - set(merged)
        if unknown:
            raise ValueError(f'Unknown importance keys: {sorted(unknown)}')
        merged.update(importance)
    for key, value in merged.items():
        if value < 0 or value > 10:
            raise ValueError(f'Importance for {key!r} must be between 0 and 10')
    total = sum(merged.values())
    if total == 0:
        raise ValueError('At least one importance value must be greater than 0')
    return {key: value / total for key, value in merged.items()}

def replacement_status_table(coords, original_labels):
    return pd.DataFrame([
        {
            'label': label,
            'status': status_for_label(coords, label),
            'action': 'keep' if status_for_label(coords, label) == 'open' else 'replace',
        }
        for label in original_labels
    ])

def run_replacement_workflow(
    coords,
    original_labels,
    top_n=10,
    nearby_pool_size=10,
    electrode_radius=0.45,
    max_replacement_distance=None,
    require_minimum_coverage=True,
    minimum_original_region_covered=85,
    importance=None,
):
    weights = importance_to_weights(importance)
    min_coverage = minimum_original_region_covered if require_minimum_coverage else None
    status_df = replacement_status_table(coords, original_labels)
    suggestions = suggest_replacement_montages(
        coords=coords,
        original_labels=original_labels,
        top_n=top_n,
        nearby_pool_size=nearby_pool_size,
        electrode_radius=electrode_radius,
        region_buffer=0.0,
        max_replacement_distance=max_replacement_distance,
        min_original_region_covered=min_coverage,
        weights=weights,
    )
    settings = {
        'top_n': top_n,
        'nearby_pool_size': nearby_pool_size,
        'electrode_radius': electrode_radius,
        'max_replacement_distance': max_replacement_distance,
        'require_minimum_coverage': require_minimum_coverage,
        'minimum_original_region_covered': minimum_original_region_covered,
        'importance': DEFAULT_IMPORTANCE.copy() if importance is None else importance,
        'weights': weights,
    }
    return {'settings': settings, 'status_table': status_df, 'suggestions': suggestions}


In [ ]:
workflow_original_labels = original_labels

workflow_importance = {
    'distance': 6,
    'pairwise_distance': 5,
    'pairwise_angle': 4,
    'region_coverage_loss': 10,
    'footprint_coverage_loss': 2,
    'new_area': 2,
    'center_shift': 2,
}

workflow_settings = {
    'top_n': 10,
    'nearby_pool_size': 10,
    'electrode_radius': 0.45,
    'max_replacement_distance': None,
    'require_minimum_coverage': True,
    'minimum_original_region_covered': 85,
}

workflow_weights = importance_to_weights(workflow_importance)
print('Normalized weights used by scorer:')
display(pd.DataFrame([workflow_weights]))


## 11. Run Wrapper Workflow

In [ ]:
try:
    workflow_results = run_replacement_workflow(
        coords=coords,
        original_labels=workflow_original_labels,
        importance=workflow_importance,
        **workflow_settings,
    )
    print('Original placement status:')
    display(workflow_results['status_table'])
    print('Ranked suggestions:')
    wrapper_suggestions_df = workflow_results['suggestions']
    display(wrapper_suggestions_df[display_columns])
except ValueError as exc:
    print(exc)
    wrapper_suggestions_df = pd.DataFrame()


## 12. Visualize Wrapper Result

In [ ]:
if 'wrapper_suggestions_df' in globals() and not wrapper_suggestions_df.empty:
    plot_original_vs_suggestion(
        coords,
        workflow_original_labels,
        wrapper_suggestions_df.iloc[0],
        title='Wrapper workflow: best suggested replacement',
    );
else:
    print('No wrapper suggestion is available to plot.')
